In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
print(os.listdir("/kaggle/input/datasets/ashishpathak778/cifar10"))

['cifar-10-batches-py']


In [3]:
import os

base_path = "/kaggle/input/datasets/ashishpathak778/cifar10/cifar-10-batches-py"
print(os.listdir(base_path))




['data_batch_1', 'data_batch_2', 'batches.meta', 'test_batch', 'data_batch_3', 'data_batch_5', 'data_batch_4', 'readme.html']


In [4]:
import torchvision
import torchvision.transforms as transforms

transform = transforms.ToTensor()

train_dataset = torchvision.datasets.CIFAR10(
    root="/kaggle/input/datasets/ashishpathak778/cifar10",
    train=True,
    download=False,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="/kaggle/input/datasets/ashishpathak778/cifar10",
    train=False,
    download=False,
    transform=transform
)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Train size: 50000
Test size: 10000


In [5]:
import pickle
import numpy as np
import time
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import SGD, Adagrad, RMSprop, Adam, Adadelta
from tensorflow.keras.utils import to_categorical

# Your base path
base_path = "/kaggle/input/datasets/ashishpathak778/cifar10/cifar-10-batches-py"


def load_batch(file):
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict


# Load training data
X_train = []
y_train = []

for i in range(1, 6):
    batch = load_batch(f"{base_path}/data_batch_{i}")
    X_train.append(batch[b'data'])
    y_train += batch[b'labels']

X_train = np.concatenate(X_train)
y_train = np.array(y_train)

# Load test data
test_batch = load_batch(f"{base_path}/test_batch")
X_test = test_batch[b'data']
y_test = np.array(test_batch[b'labels'])

# Reshape
X_train = X_train.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
X_test = X_test.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)

# Normalize
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# One-hot encoding
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

2026-03-26 13:28:39.322990: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774531719.544008      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774531719.611362      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774531720.210934      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774531720.210966      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774531720.210969      55 computation_placer.cc:177] computation placer alr

Train size: (50000, 32, 32, 3)
Test size: (10000, 32, 32, 3)


In [6]:
def build_mlp(optimizer):
    model = Sequential([
        Flatten(input_shape=(32, 32, 3)),
        Dense(512, activation='relu'),
        Dense(256, activation='relu'),
        Dense(10, activation='softmax')
    ])
    
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [7]:
results = {}

def train_and_evaluate(name, optimizer, batch_size):
    print(f"\nTraining with {name}...")
    model = build_mlp(optimizer)
    
    start = time.time()
    model.fit(X_train, y_train, epochs=10, batch_size=batch_size, verbose=0)
    end = time.time()
    
    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    
    results[name] = {
        "Accuracy": acc,
        "Training Time (s)": end - start
    }


# Batch GD
train_and_evaluate("Batch GD", SGD(learning_rate=0.01), batch_size=50000)

# SGD
train_and_evaluate("Stochastic GD", SGD(learning_rate=0.01), batch_size=1)

# Mini-batch GD
train_and_evaluate("Mini-batch GD", SGD(learning_rate=0.01), batch_size=32)

# Momentum
train_and_evaluate("Momentum GD", SGD(learning_rate=0.01, momentum=0.9), batch_size=32)

# Nesterov
train_and_evaluate("Nesterov GD", SGD(learning_rate=0.01, momentum=0.9, nesterov=True), batch_size=32)

# Adagrad
train_and_evaluate("Adagrad", Adagrad(learning_rate=0.01), batch_size=32)

# RMSprop
train_and_evaluate("RMSprop", RMSprop(learning_rate=0.001), batch_size=32)

# Adadelta
train_and_evaluate("Adadelta", Adadelta(), batch_size=32)

# Adam
train_and_evaluate("Adam", Adam(learning_rate=0.001), batch_size=32)

I0000 00:00:1774531747.301876      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774531747.307992      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Training with Batch GD...


I0000 00:00:1774531751.954449     120 service.cc:152] XLA service 0x7a75c0009270 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774531751.954500     120 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1774531751.954506     120 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1774531752.150619     120 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1774531757.537946     120 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



Training with Stochastic GD...

Training with Mini-batch GD...

Training with Momentum GD...

Training with Nesterov GD...

Training with Adagrad...

Training with RMSprop...

Training with Adadelta...

Training with Adam...


In [8]:
df_results = pd.DataFrame(results).T
df_results

,Accuracy,Training Time (s)
Batch GD,0.1834,15.374067
Stochastic GD,0.4080,924.249521
Mini-batch GD,0.4809,35.017332
Momentum GD,0.4826,35.875367
Nesterov GD,0.4775,36.423023
Adagrad,0.5152,36.154675
RMSprop,0.4472,36.250381
Adadelta,0.3553,38.332934
Adam,0.4774,38.054559


In [9]:
def build_base_mlp():
    model = Sequential([
        Flatten(input_shape=(32, 32, 3)),
        Dense(512, activation='relu'),
        Dense(256, activation='relu'),
        Dense(10, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

****L2 Regularization****

In [10]:
from tensorflow.keras.regularizers import l2

def build_l2_mlp():
    model = Sequential([
        Flatten(input_shape=(32, 32, 3)),
        Dense(512, activation='relu', kernel_regularizer=l2(0.001)),
        Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
        Dense(10, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

**Dropout**

In [11]:
from tensorflow.keras.layers import Dropout

def build_dropout_mlp():
    model = Sequential([
        Flatten(input_shape=(32, 32, 3)),
        Dense(512, activation='relu'),
        Dropout(0.5),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(10, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

**Dataset Augmentation**

In [12]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

datagen.fit(X_train)

In [ ]:
model_aug = build_base_mlp()
model_aug.fit(datagen.flow(X_train, y_train, batch_size=32),
              epochs=10,
              validation_data=(X_test, y_test))

Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1563/1563 ━━━━━━━━━━━━━━━━━━━━ 30s 18ms/step - accuracy: 0.2583 - loss: 2.0552 - val_accuracy: 0.3718 - val_loss: 1.7520
Epoch 2/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - accuracy: 0.3573 - loss: 1.7835 - val_accuracy: 0.4013 - val_loss: 1.6644
Epoch 3/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - accuracy: 0.3857 - loss: 1.7057 - val_accuracy: 0.4396 - val_loss: 1.5744
Epoch 4/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - accuracy: 0.4032 - loss: 1.6690 - val_accuracy: 0.4453 - val_loss: 1.5520
Epoch 5/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 27s 18ms/step - accuracy: 0.4076 - loss: 1.6449 - val_accuracy: 0.4439 - val_loss: 1.5539
Epoch 6/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - accuracy: 0.4158 - loss: 1.6328 - val_accuracy: 0.4250 - val_loss: 1.5802
Epoch 7/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 27s 18ms/step - accuracy: 0.4257 - loss: 1.5969 - val_accuracy: 0.4380 - val_loss: 1.5798


**Adding Noise to Inputs**

In [ ]:
from tensorflow.keras.layers import GaussianNoise

def build_noise_mlp():
    model = Sequential([
        Flatten(input_shape=(32, 32, 3)),
        GaussianNoise(0.1),
        Dense(512, activation='relu'),
        Dense(256, activation='relu'),
        Dense(10, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

**Early Stopping**

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

model_es = build_base_mlp()
model_es.fit(X_train, y_train,
             epochs=50,
             batch_size=32,
             validation_split=0.2,
             callbacks=[early_stop])

**Ensemble Methods**

In [ ]:
models = []
for i in range(3):
    model = build_base_mlp()
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    models.append(model)

# Ensemble prediction
preds = np.mean([model.predict(X_test) for model in models], axis=0)
ensemble_acc = np.mean(np.argmax(preds, axis=1) == np.argmax(y_test, axis=1))
print("Ensemble Accuracy:", ensemble_acc)

Parameter Sharing and Tying (MLP Adaptation)

In [ ]:
shared_dense = Dense(256, activation='relu')

def build_shared_mlp():
    model = Sequential([
        Flatten(input_shape=(32, 32, 3)),
        shared_dense,
        shared_dense,
        Dense(10, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

**Create Comparision Function**

In [ ]:
results_reg = {}

def train_and_compare(name, model):
    print(f"\nTraining: {name}")
    
    history = model.fit(
        X_train, y_train,
        epochs=15,
        batch_size=32,
        validation_split=0.2,
        verbose=0
    )
    
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    
    results_reg[name] = {
        "Train Accuracy": history.history['accuracy'][-1],
        "Val Accuracy": history.history['val_accuracy'][-1],
        "Test Accuracy": test_acc
    }

**Base Model**

In [ ]:
train_and_compare("Base MLP", build_base_mlp())

**L2 Regularization**

In [ ]:
train_and_compare("L2 Regularization", build_l2_mlp())

**Dropout**

In [ ]:
train_and_compare("Dropout", build_dropout_mlp())

**Noise**

In [ ]:
train_and_compare("Input Noise", build_noise_mlp())

**Early Stopping**

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

model_es = build_base_mlp()

history = model_es.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0
)

test_loss, test_acc = model_es.evaluate(X_test, y_test, verbose=0)

results_reg["Early Stopping"] = {
    "Train Accuracy": history.history['accuracy'][-1],
    "Val Accuracy": history.history['val_accuracy'][-1],
    "Test Accuracy": test_acc
}

**Dataset Augmentation**

In [ ]:
model_aug = build_base_mlp()

history = model_aug.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=15,
    validation_data=(X_test, y_test),
    verbose=0
)

test_loss, test_acc = model_aug.evaluate(X_test, y_test, verbose=0)

results_reg["Data Augmentation"] = {
    "Train Accuracy": history.history['accuracy'][-1],
    "Val Accuracy": history.history['val_accuracy'][-1],
    "Test Accuracy": test_acc
}

**Ensemble**

In [ ]:
models = []
for i in range(3):
    model = build_base_mlp()
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    models.append(model)

preds = np.mean([m.predict(X_test) for m in models], axis=0)
ensemble_acc = np.mean(np.argmax(preds, axis=1) == np.argmax(y_test, axis=1))

results_reg["Ensemble"] = {
    "Train Accuracy": "—",
    "Val Accuracy": "—",
    "Test Accuracy": ensemble_acc
}

In [ ]:
import pandas as pd

df_reg = pd.DataFrame(results_reg).T
df_reg

In [ ]:
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()

In [ ]:
from tensorflow.keras.utils import to_categorical

# Normalize
X_train = X_train.astype('float32') / 255
X_test = X_test.astype('float32') / 255

# One-hot encoding
n_classes = 10
Y_train = to_categorical(y_train, n_classes)
Y_test = to_categorical(y_test, n_classes)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv2D, MaxPool2D, Flatten

model = Sequential()

# First Convolution Block
model.add(Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)))
model.add(MaxPool2D((2,2)))

# Second Convolution Block
model.add(Conv2D(64, (3,3), activation='relu', padding='same'))
model.add(MaxPool2D((2,2)))

# Flatten
model.add(Flatten())

# Fully Connected Layer
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

# Output Layer
model.add(Dense(10, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    X_train, Y_train,
    batch_size=128,
    epochs=15,
    validation_data=(X_test, Y_test)
)

In [ ]:
test_loss, test_acc = model.evaluate(X_test, Y_test)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_acc)

In [ ]:
from keras.datasets import cifar10
from keras.models import Sequential, Model
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from keras.layers import Input, Add, BatchNormalization
from tensorflow.keras.utils import to_categorical


In [ ]:
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

X_train = X_train / 255.0
X_test = X_test / 255.0

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

results = {}


In [ ]:
lenet_model = Sequential([
    Conv2D(6,(5,5),activation='relu',input_shape=(32,32,3)),
    MaxPooling2D(),
    Conv2D(16,(5,5),activation='relu'),
    MaxPooling2D(),
    Flatten(),
    Dense(120,activation='relu'),
    Dense(84,activation='relu'),
    Dense(10,activation='softmax')
])

lenet_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
lenet_model.fit(X_train, y_train, epochs=5, batch_size=64, verbose=0)

_, acc = lenet_model.evaluate(X_test, y_test, verbose=0)
results['LeNet'] = acc


In [ ]:
alex_model = Sequential([
    Conv2D(96,(3,3),activation='relu',input_shape=(32,32,3)),
    MaxPooling2D(),
    Conv2D(256,(3,3),activation='relu'),
    MaxPooling2D(),
    Conv2D(384,(3,3),activation='relu'),
    Flatten(),
    Dense(256,activation='relu'),
    Dropout(0.5),
    Dense(10,activation='softmax')
])

alex_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
alex_model.fit(X_train, y_train, epochs=5, batch_size=64, verbose=0)

_, acc = alex_model.evaluate(X_test, y_test, verbose=0)
results['AlexNet'] = acc

In [ ]:
zf_model = Sequential([
    Conv2D(96,(3,3),activation='relu',input_shape=(32,32,3)),
    MaxPooling2D(),
    Conv2D(256,(3,3),activation='relu'),
    MaxPooling2D(),
    Conv2D(384,(3,3),activation='relu'),
    Flatten(),
    Dense(256,activation='relu'),
    Dense(10,activation='softmax')
])

zf_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
zf_model.fit(X_train, y_train, epochs=5, batch_size=64, verbose=0)

_, acc = zf_model.evaluate(X_test, y_test, verbose=0)
results['ZF-Net'] = acc

In [ ]:
vgg_model = Sequential([
    Conv2D(64,(3,3),activation='relu',padding='same',input_shape=(32,32,3)),
    Conv2D(64,(3,3),activation='relu',padding='same'),
    MaxPooling2D(),

    Conv2D(128,(3,3),activation='relu',padding='same'),
    Conv2D(128,(3,3),activation='relu',padding='same'),
    MaxPooling2D(),

    Flatten(),
    Dense(256,activation='relu'),
    Dense(10,activation='softmax')
])

vgg_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
vgg_model.fit(X_train, y_train, epochs=5, batch_size=64, verbose=0)

_, acc = vgg_model.evaluate(X_test, y_test, verbose=0)
results['VGGNet'] = acc

In [ ]:
input_layer = Input(shape=(32,32,3))

c1 = Conv2D(32,(1,1),activation='relu',padding='same')(input_layer)
c3 = Conv2D(32,(3,3),activation='relu',padding='same')(input_layer)
c5 = Conv2D(32,(5,5),activation='relu',padding='same')(input_layer)

merge = MaxPooling2D()(c1 + c3 + c5)
flat = Flatten()(merge)
dense = Dense(128,activation='relu')(flat)
output = Dense(10,activation='softmax')(dense)

google_model = Model(inputs=input_layer, outputs=output)

google_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
google_model.fit(X_train, y_train, epochs=5, batch_size=64, verbose=0)

_, acc = google_model.evaluate(X_test, y_test, verbose=0)
results['GoogLeNet'] = acc

In [ ]:
input_layer = Input(shape=(32,32,3))

x = Conv2D(32,(3,3),padding='same',activation='relu')(input_layer)

shortcut = x
x = Conv2D(32,(3,3),padding='same',activation='relu')(x)
x = Conv2D(32,(3,3),padding='same')(x)

x = Add()([x, shortcut])
x = BatchNormalization()(x)

x = MaxPooling2D()(x)
x = Flatten()(x)
x = Dense(128,activation='relu')(x)
output = Dense(10,activation='softmax')(x)

resnet_model = Model(inputs=input_layer, outputs=output)

resnet_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
resnet_model.fit(X_train, y_train, epochs=5, batch_size=64, verbose=0)

_, acc = resnet_model.evaluate(X_test, y_test, verbose=0)
results['ResNet'] = acc

In [ ]:
for model_name, acc in results.items():
    print(model_name, "Accuracy:", acc)